## Preamble

In [ ]:
# Install packages if needed
# install.packages("tidyverse")
# install.packages("WDI")
# install.packages("RSQLite")

# Load libraries
library("tidyverse")
library("WDI")
library("RSQLite")

options(repr.plot.width = 12, repr.plot.height = 8)

In [ ]:
# Directory
setwd('/Users/tasia/Desktop/Data Analysis/Assessment')

### Load and Set Up GDP and Unemployment Data

In [ ]:
# Load data from database
conn <- dbConnect(SQLite(), "wdi_cross_sectional_gdp_unemployment_2023.db")

df_gdp <- dbGetQuery(conn, "SELECT * FROM gdp")
df_unp <- dbGetQuery(conn, "SELECT * FROM unemployment")

dbDisconnect(conn)

# Eyeball
head(df_gdp)
head(df_unp)

In [ ]:
# Convert columns to numeric and replace invalid values with NaN
df_gdp$`2023 [YR2023]` <- as.numeric(na_if(df_gdp$`2023 [YR2023]`, ".."))
df_unp$`2023 [YR2023]` <- as.numeric(na_if(df_unp$`2023 [YR2023]`, ".."))

# Eyeball
head(df_gdp)
head(df_unp)

In [ ]:
# Remove rows with missing country codes
df_gdp <- filter(df_gdp, !is.na(`Country Code`), `Country Code` != "")
df_unp <- filter(df_unp, !is.na(`Country Code`), `Country Code` != "")

# Merge dataframes (inner join!)
df <- inner_join(df_gdp,
                 df_unp,
                 by = "Country Code")

# Eyeball
head(df)

In [ ]:
# Rename columns
df <- rename(df,
             gdp = `2023 [YR2023].x`,
             unemployment = `2023 [YR2023].y`)

# Eyeball
head(df)

In [ ]:
# Load country information
countries <- WDI_data$country

# Remove World Bank aggregates
countries <- filter(countries, region != "Aggregates")
df <- filter(df, `Country Code` %in% countries$iso3c)

# Remove missing values
df <- drop_na(df, gdp, unemployment)

# Add region information
df <- left_join(df,
                select(countries, iso3c, region),
                by = c("Country Code" = "iso3c"))

# Eyeball
head(df)

## Running 4 Regression Models

### Model 1: Simple Linear Regression

In [ ]:
# Regress unemployment on GDP per capita
model1 <- lm(unemployment ~ gdp, data = df)
summary(model1)

### Interpretation and Discussing the Output

Interpretation and Discussing the Output

### Model 2: Linear-Log Regression

In [ ]:
# Regress unemployment on logged GDP per capita
model2 <- lm(unemployment ~ log(gdp), data = df)
summary(model2)

### Interpretation and Discussing the Output

Interpretation and Discussing the Output

### Model 3: Quadratic Regression

In [ ]:
# Regress unemployment on GDP per capita and GDP per capita squared
model3 <- lm(unemployment ~ gdp + I(gdp^2), data = df)
summary(model3)

### Interpretation and Discussing the Output

Interpretation and Discussing the Output

### Model 4: Multiple Linear Regression with Regional Controls

In [ ]:
# Regress unemployment on GDP per capita and region
model4 <- lm(unemployment ~ gdp + region, data = df)
summary(model4)

### Interpretation and Discussing the Output

Interpretation and Discussing the Output